# Colab bootstrap — realized-volatility-prediction

Thin launcher, not an experiment notebook (manifesto: shared `src/` library, thin notebooks). Clones the repo, installs the delta packages Colab doesn't already ship, then shells out to the Stage 1 CLI driver (`scripts/run_stage1.py`). All logic lives in `src/`; this notebook only wires up the environment.

Re-run the clone cell any time to pull the latest `main` before a new run.

In [ ]:
import os

REPO_URL = "https://github.com/Stridasaurus/realized-volatility-prediction.git"
REPO_DIR = "realized-volatility-prediction"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

In [ ]:
# Unpinned deltas only — Colab already ships numpy/pandas/scipy/torch, and hard-pinning
# the local dev env's versions here risks forcing a slow/breaking reinstall of that stack.
!pip install -q -r requirements.txt

In [ ]:
!PYTHONPATH=. python -m pytest -q

## Stage 1: tune

Optuna search on TV/h=1, train/val only. Writes the frozen HP set into `configs/default.yaml` (review the diff and commit it — that commit pre-registers the frozen control per manifesto s7). Use `--dry-run` first to preview without touching the config.

In [ ]:
!python scripts/run_stage1.py tune --n-trials 50 --dry-run

## Stage 1: compare

RV-only LSTM vs HAR, walk-forward over the frozen retrain folds, both TV horizons. Reads whatever HP set is currently committed in `configs/default.yaml` — run `tune` (without `--dry-run`) and commit the config change first.

In [ ]:
!python scripts/run_stage1.py compare --seeds headline